<a href="https://colab.research.google.com/github/hamsamahmoud-cpu/DistilBERT-Fine-tuning-for-Movie-Review-Sentiment-Analysis1/blob/main/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the core libraries
!pip install -U transformers datasets evaluate accelerate huggingface_hub

In [ ]:
from transformers import pipeline

# This automatically downloads the default sentiment model and tokenizer
classifier = pipeline("sentiment-analysis")

# Test it
result = classifier("I am so excited to start Project 15!")
print(result)
# Expected Output: [{'label': 'POSITIVE', 'score': 0.999}]

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# This pulls the token from your Colab Secrets safely
hf_token = userdata.get('HF_TOKEN')

# Log into the Hugging Face Hub
login(token=hf_token)

In [ ]:
# Install core libraries
!pip install -U transformers datasets evaluate accelerate huggingface_hub

import os
from google.colab import drive, userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Mount Google Drive for saving results
drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/distilbert-imdb-sentiment'
os.makedirs(PROJECT_PATH, exist_ok=True)

# 2. Login using your HF Token (Assumes you saved it in Colab Secrets as 'HF_TOKEN')
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub.")
except Exception as e:
    print("Login failed. Make sure 'HF_TOKEN' is in your Colab Secrets (key icon).")

In [ ]:
# Load the IMDB dataset
print("Downloading IMDB dataset...")
raw_datasets = load_dataset("imdb")

# Initialize the DistilBERT tokenizer
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

print(f"Tokenizer loaded: {model_ckpt}")

In [ ]:
def tokenize_function(examples):
    # This turns text into input_ids and attention_masks
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

print("Mapping tokenization across the dataset (this may take 5 mins)...")
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Post-processing: remove raw text and prepare for PyTorch
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

In [ ]:
# Verify the shape: Expected [512] for each sample
sample_ids = tokenized_datasets["train"][0]["input_ids"]
print(f"Verification: First review token length is {len(sample_ids)}")

# Save to Disk
save_path = os.path.join(PROJECT_PATH, "tokenized_data")
tokenized_datasets.save_to_disk(save_path)
print(f"Week 1 Complete! Data saved to: {save_path}")